#   Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

#   Load the dataset

In [ ]:
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
display(df.head())

In [ ]:
print(df.info())

#   Data Cleaning

In [ ]:
# The 'TotalCharges' column is imported as an object (string) due to blank spaces for new customers.
# We must convert it to a numeric data type. 'coerce' turns blank spaces into NaN.
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [ ]:
# Check how many NaN values were created
print(f"Missing values in TotalCharges: {df['TotalCharges'].isnull().sum()}")

In [ ]:
# Since the missing values are very few (around 11 out of 7000+), it is safe to drop them
df.dropna(inplace=True)

In [ ]:
# Drop the 'customerID' column because unique identifiers do not hold predictive power for ML models
df.drop(columns=['customerID'], inplace=True)

In [ ]:
# 3. Target Variable Encoding
# Convert the target variable 'Churn' from string ('Yes'/'No') to binary integer (1/0)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

In [ ]:
# 4. Check Class Distribution
# It is critical to see if our dataset is imbalanced
class_counts = df['Churn'].value_counts()
print("\nClass Distribution:")
print(class_counts)

#   Visualize the distribution

In [ ]:
sns.countplot(x='Churn', data=df, palette='viridis')
plt.title('Customer Churn Distribution')
plt.show()

#   Categorical Data Encoding

In [ ]:
# Convert all categorical text columns into numerical dummy variables (0 or 1)
# drop_first=True removes the first level to avoid the dummy variable trap (multicollinearity)
df_encoded = pd.get_dummies(df, drop_first=True)

In [ ]:
# 6. Define Features (X) and Target Variable (y)
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

#   Data Spliting And Model Building

In [ ]:
# 7. Split the Data into Training and Testing Sets
# stratify=y ensures both training and testing sets have the same proportion of Churn (1) and No Churn (0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

In [ ]:
# 8. Model Building: Random Forest Classifier
# Initialize the model
# class_weight='balanced' is crucial here! It automatically adjusts weights to penalize 
# mistakes on the minority class (Churn = 1) because our dataset is imbalanced.
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')

In [ ]:
# Train the model
rf_model.fit(X_train, y_train)

In [ ]:
# 9. Model Evaluation
rf_predictions = rf_model.predict(X_test)

print("\n--- Random Forest Classification Report ---")
print(classification_report(y_test, rf_predictions))

#   Feature Importance: Understanding WHY customers churn

In [ ]:
# Extract importance scores from the Random Forest model
importances = rf_model.feature_importances_
features = X.columns

In [ ]:
# Create a DataFrame to sort the features by their importance
importance_df = pd.DataFrame({'Feature': features, 'Importance': importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

In [ ]:
# Plot the top 10 most critical features driving churn
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importance_df.head(10), palette='magma')
plt.title('Top 10 Features Driving Customer Churn')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()